# KodaLite-1.3B quickstart

Sanity check the codebase end to end: architecture, tokenizer, a tiny pretrain step, a tiny SFT LoRA step, and loading the released checkpoint from HuggingFace.

Run from the repo root (so `model.py`, `lora.py`, etc. are importable). Add the parent dir to `sys.path` if you launched Jupyter from inside `notebooks/`.

All phases below run on a single GPU at small scale.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.abspath('..'))
import jax
print('jax devices:', jax.devices())

## 1. Architecture sanity check

Build the `xl` config (1.27B params), count parameters, run a forward pass on dummy tokens. Drop to a `small` config if VRAM is tight.

In [ ]:
import jax.numpy as jnp
import flax.nnx as nnx
import tiktoken
from model import MiniGPT, CONFIGS

tokenizer = tiktoken.get_encoding('gpt2')
config = CONFIGS['xl'].copy()  # change to 'small' / 'medium' for less VRAM
config['vocab_size'] = tokenizer.n_vocab
print('config:', config)

model = MiniGPT(config, dtype=jnp.bfloat16, use_gradient_checkpointing=False, rngs=nnx.Rngs(0))
n_params = sum(int(jnp.size(p)) for p in jax.tree_util.tree_leaves(nnx.state(model, nnx.Param)))
print(f'params: {n_params/1e9:.3f}B')

x = jnp.array([tokenizer.encode('Hello world')], dtype=jnp.int32)
logits = model(x, deterministic=True)
print('logits shape:', logits.shape, 'dtype:', logits.dtype)

## 2. Tiny pretrain step

Stream a few SlimPajama batches and run one optax step. This is the same code path as `train.py`, just stopped after a couple steps.

In [ ]:
import optax
from data import StreamingDataLoader

loader = StreamingDataLoader(maxlen=config['maxlen'], batch_size=2, source='slimpajama')
optimizer = nnx.Optimizer(model, optax.adamw(1e-4, weight_decay=0.1))

def loss_fn(model, tokens, targets):
    logits = model(tokens, deterministic=False)
    return optax.softmax_cross_entropy_with_integer_labels(logits, targets).mean()

for step in range(2):
    batch = next(loader)
    tokens = jnp.asarray(batch[:, :-1])
    targets = jnp.asarray(batch[:, 1:])
    loss, grads = nnx.value_and_grad(loss_fn)(model, tokens, targets)
    optimizer.update(grads)
    print(f'step {step} loss={float(loss):.3f}')

## 3. Inject LoRA and run a tiny SFT step

Wraps every attention projection in a LoRA adapter. Only the LoRA params are trainable, the base is frozen.

In [ ]:
from lora import inject_lora, count_lora_params
from sft_loader_eos import SFTDataLoaderEOS

model = inject_lora(model, rank=16, alpha=32.0, rngs=nnx.Rngs(42))
lora_n, total_n = count_lora_params(model)
print(f'LoRA: {lora_n/1e6:.2f}M trainable, total {total_n/1e9:.3f}B')

sft = SFTDataLoaderEOS(maxlen=config['maxlen'], batch_size=2)
batch = sft.next_batch()
print('batch shapes:', {k: v.shape for k, v in batch.items()})

## 4. Load the released checkpoint and chat

Pulls the EOS-fixed weights from HuggingFace and runs a single greedy completion. No GPU needed, MLX 8bit runs comfortably on a Mac.

In [ ]:
# Apple Silicon (MLX, no GPU needed):
# pip install mlx-lm
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

m, t = load('YoAbriel/KodaLite-1.3B-mlx-8bit')
prompt = t.apply_chat_template(
    [{'role': 'user', 'content': 'What is the capital of France?'}],
    tokenize=False,
)
print(generate(m, t, prompt=prompt, max_tokens=120, sampler=make_sampler(temp=0.0)))

In [ ]:
# CUDA (HF Transformers):
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# tok = AutoTokenizer.from_pretrained('YoAbriel/KodaLite-1.3B')
# model = AutoModelForCausalLM.from_pretrained('YoAbriel/KodaLite-1.3B', dtype=torch.bfloat16, device_map='auto')
# msg = [{'role': 'user', 'content': 'What is the capital of France?'}]
# inputs = tok.apply_chat_template(msg, return_tensors='pt', add_generation_prompt=False).to(model.device)
# out = model.generate(inputs, max_new_tokens=120, do_sample=False)
# print(tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))